# Logistics Late Orders Prediction

This notebook outlines the end-to-end Machine Learning pipeline used to predict the probability of a logistics order arriving late. The process includes data cleaning, feature engineering, and training an XGBoost classifier optimized with GridSearchCV.

In [ ]:
# 1. Import required libraries
import numpy as np
import pandas as pd
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score

# 2. Load all datasets
df_orders = pd.read_csv("orders.csv", sep=";")
df_cities = pd.read_csv("cities_data.csv", sep=";")
df_cities_costs = pd.read_csv("cities_data_costs.csv", sep=";")
df_products_attributes = pd.read_csv("product_attributes.csv", sep=";")
df_product_weight_class = pd.read_csv("product_weight_class.csv", sep=";")
df_test = pd.read_csv("test.csv", sep=";")

# 3. Prepare the cities dataset (Enable bidirectional routes)
df_cities_reversed = df_cities.rename(columns={"city_to_name": "city_from_name", "city_from_name": "city_to_name"})
df_cities_fixed = pd.concat([df_cities, df_cities_reversed], ignore_index=False)

## Training Data Preprocessing (Train)
We will correct inconsistencies in city names, merge datasets to calculate the distances for both legs of the journey (Origin -> Hub, and Hub -> Customer), and encode categorical variables for our model.

In [ ]:
# Correct city names
df_orders['origin_port'] = df_orders['origin_port'].replace({"ATHENAS": "Athens", "BCN": "Barcelona"})

# Merge 1: Calculate distance from Origin Port to Logistic Hub
df_orders = pd.merge(df_orders, df_cities_fixed, left_on=['origin_port', 'logistic_hub'], right_on=['city_from_name', 'city_to_name'], how='left')
df_orders = df_orders.drop(columns=['origin_port', 'city_from_name', 'city_to_name'])
df_orders = df_orders.rename(columns={'city_from_coord': 'origin_port', 'city_to_coord': 'logistic_hub1', 'distance': 'distance1'})

# Merge 2: Calculate distance from Logistic Hub to Final Customer
df_orders = pd.merge(df_orders, df_cities_fixed, left_on=['logistic_hub', 'customer'], right_on=['city_from_name', 'city_to_name'], how='left')
df_orders = df_orders.drop(columns=['logistic_hub', 'customer', 'city_from_name', 'city_to_name', 'city_from_coord'])
df_orders = df_orders.rename(columns={'city_to_coord': 'customer', 'logistic_hub1': 'logistic_hub', 'distance': 'distance2'})

# Encode categorical variables mapping them to integers
df_orders['3pl'] = df_orders['3pl'].replace({"v_001": 1, "v_002": 2, "v_003": 3, "v_004": 4})
df_orders['customs_procedures'] = df_orders['customs_procedures'].replace({"DTP": 1, "CRF": 2, "DTD": 3})

# Drop non-numeric coordinate columns and handle missing values
df_orders_no_coord = df_orders.drop(columns=['origin_port', 'logistic_hub', 'customer'])
df_orders_no_coord = df_orders_no_coord.fillna(0)

# Optional: Keep track of specific variables for EDA (Exploratory Data Analysis)
df_late_order = df_orders['late_order']
df_order_id = df_orders['order_id']

# Separate features and target, then save clean data locally
df_train = df_orders_no_coord.drop(columns=['late_order', 'order_id'])
df_train.to_csv('x_train.csv', index=False)

df_y_train = df_orders_no_coord['late_order']
df_y_train.to_csv('y_train.csv', index=False)

## Test Data Preprocessing (Test)
To ensure the model predicts accurately, we must apply the exact same transformation and cleaning pipeline to the `test.csv` dataset.

In [ ]:
# Correct city names
df_test['origin_port'] = df_test['origin_port'].replace({"ATHENAS": "Athens", "BCN": "Barcelona"})

# Merge 1: Port -> Hub
df_test = pd.merge(df_test, df_cities_fixed, left_on=['origin_port', 'logistic_hub'], right_on=['city_from_name', 'city_to_name'], how='left')
df_test = df_test.drop(columns=['origin_port', 'city_from_name', 'city_to_name'])
df_test = df_test.rename(columns={'city_from_coord': 'origin_port', 'city_to_coord': 'logistic_hub1', 'distance': 'distance1'})

# Merge 2: Hub -> Customer
df_test = pd.merge(df_test, df_cities_fixed, left_on=['logistic_hub', 'customer'], right_on=['city_from_name', 'city_to_name'], how='left')
df_test = df_test.drop(columns=['logistic_hub', 'customer', 'city_from_name', 'city_to_name', 'city_from_coord'])
df_test = df_test.rename(columns={'city_to_coord': 'customer', 'logistic_hub1': 'logistic_hub', 'distance': 'distance2'})

# Encode categorical variables
df_test['3pl'] = df_test['3pl'].replace({"v_001": 1, "v_002": 2, "v_003": 3, "v_004": 4})
df_test['customs_procedures'] = df_test['customs_procedures'].replace({"DTP": 1, "CRF": 2, "DTD": 3})

# Drop coordinates and fill nulls
df_test_no_coord = df_test.drop(columns=['origin_port', 'logistic_hub', 'customer'])
df_test_no_coord = df_test_no_coord.fillna(0)

## Model Training & Prediction
We will train an `XGBClassifier` and use `GridSearchCV` to find the optimal combination of tree depth and the number of estimators. Finally, we generate the predictions and format them for Kaggle submission.

In [ ]:
# Define features (X) and target (y) for the training set
x_train = df_orders_no_coord.drop(columns=['late_order', 'order_id'])
y_train = df_orders_no_coord['late_order']

# Define features for the test set
x_test = df_test_no_coord.drop(columns=['order_id'])

# Initialize XGBoost model and define hyperparameter grid
xgb_model = xgb.XGBClassifier()
optimization_dict = {
    'max_depth': [4, 8, 12],
    'n_estimators': [200, 400, 800]
}

# Perform Grid Search to find the best model
model = GridSearchCV(xgb_model, optimization_dict, scoring='accuracy', verbose=1)
model.fit(x_train, y_train)

# Generate probability predictions for the test set
pred_proba = model.predict_proba(x_test)

# Format the final submission dataframe
submission = pd.DataFrame({
    "order_id": df_test_no_coord['order_id'],
    "late_order": pred_proba[:, 1] # Index 1 extracts the probability of the positive class (being late)
})

# Export results to CSV
submission.to_csv("submission_kaggle.csv", index=False)